In [ ]:
# [notebook_constants]

USE_KAGGLE_SANDBOX = True
KAGGLE_SANDBOX_KWARGS = {
    'dataset_data_sources': ['greynearing/camels-us-531-era'],
    'enable_gpu': True,
    'enable_internet': False,
    'machine_shape': 'NvidiaTeslaT4Highmem',
}
C_PUCT = 2.5

# CAMELS-US ERA — Gauged Minimal Guidance

## Objective
Maximize the **Median Nash-Sutcliffe Efficiency (NSE)** across all **531 CAMELS-US basins** on the out-of-sample evaluation period (`1999-10-01` to `2003-09-30`, 4 water years, `1,461` days).

## Data & Split Protocol
- **Spatial Regime (Gauged)**: All 531 basins have training streamflow observations `QOBS_TRAIN` (`mm/day`) available during the weights training period (`2003-10-01` to `2008-09-30`, 5 water years, `1,827` days).
- **Strict Newman Temporal Containment (`1998-10-01` to `2008-09-30`, `3,653` days)**:
  - `1998-10-01` to `1999-09-30` (`365` days): 1-year meteorological spin-up prior to evaluation (`QOBS_TRAIN = NaN`).
  - `1999-10-01` to `2003-09-30` (`1,461` days, `ERA_EVAL_MASK`): Out-of-sample ERA hill-climb evaluation period (`QOBS_TRAIN = NaN`). The final 365 days (`2002-10-01` to `2003-09-30`) also serve as the 1-year meteorological spin-up prior to `2003-10-01`.
  - `2003-10-01` to `2008-09-30` (`1,827` days, `ERA_TRAIN_MASK`): ERA model weights training period (`QOBS_TRAIN` observed).
- **Inputs Available**:
  - `DYNAMIC_INPUTS`: shape `(531, 3653, 15)` (`float32`) — 5 daily meteorological variables (`prcp`, `srad`, `tmax`, `tmin`, `vp`) across 3 forcing products (`Daymet`, `Maurer`, `NLDAS`).
  - `AUX_FORCINGS`: shape `(531, 3653, 6)` (`float32`) — daily `dayl` and `swe` across `Daymet`, `Maurer`, `NLDAS`.
  - `STATIC_ATTRIBUTES`: shape `(531, 27)` (`float32`) — 27 catchment climate, topography, soil, geology, and vegetation attributes (plus `BASIN_AREAS`, `LAT`, `LON`, `HUC_02`).
- **Required Output**:
  - The mutable cell must produce `Q_PRED` as a `float32` NumPy array of shape `(531, 3653)` containing daily streamflow predictions (`mm/day`).

## Modeling & Exploration Scope
You are free to explore any machine learning architecture, sequence length, feature representation, normalization strategy, loss function, regularization, optimizer schedule, or multi-model ensembling strategy, provided all predictions strictly respect the causal temporal masks and spatial fold boundaries.

In [ ]:
import os
import time
import math
import json
import random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Ensure reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0

_SEARCH_DIRS = [
    Path("/kaggle/input/camels-us-531-era"),
    Path("/kaggle/input/datasets/greynearing/camels-us-531-era"),
    Path(__file__).resolve().parent.parent / "splits" if "__file__" in globals() else Path("../splits"),
    Path("../splits"),
    Path("./splits"),
]

def _find_file(filename: str) -> Path:
    for d in _SEARCH_DIRS:
        candidate = d / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not locate {filename} in any of {_SEARCH_DIRS}")

_train_npz = np.load(_find_file("camels_era_train.npz"), allow_pickle=True)
_basins_json = json.loads(_find_file("basins_531.json").read_text(encoding="utf-8"))

BASINS = _train_npz["basins"].astype(str)
_canonical_basins = [str(b) for b in _basins_json]
assert list(BASINS) == _canonical_basins, "BASINS order mismatch against canonical basins_531.json"

# Strictly enforce containment to the Newman training window (1998-10-01 to 2008-09-30, 3653 days)
_raw_dates = np.array([str(d) for d in _train_npz["dates"]])
_newman_window = (_raw_dates >= "1998-10-01") & (_raw_dates <= "2008-09-30")
DATES = _raw_dates[_newman_window]
assert len(DATES) == 3653 and DATES[0] == "1998-10-01" and DATES[-1] == "2008-09-30"

DYNAMIC_INPUTS = _train_npz["dynamic_inputs"][:, _newman_window, :].astype(np.float32)
DYNAMIC_FEATURE_NAMES = [str(x) for x in _train_npz["dynamic_feature_names"]]
AUX_FORCINGS = _train_npz["aux_forcings"][:, _newman_window, :].astype(np.float32)
AUX_FEATURE_NAMES = [str(x) for x in _train_npz["aux_feature_names"]]
STATIC_ATTRIBUTES = _train_npz["static_attributes"].astype(np.float32)
STATIC_ATTRIBUTE_NAMES = [str(x) for x in _train_npz["static_attribute_names"]]

ERA_TRAIN_MASK = (DATES >= "2003-10-01") & (DATES <= "2008-09-30")
ERA_EVAL_MASK = (DATES >= "1999-10-01") & (DATES <= "2003-09-30")
assert int(ERA_TRAIN_MASK.sum()) == 1827, f"Expected 1827 training days, got {int(ERA_TRAIN_MASK.sum())}"
assert int(ERA_EVAL_MASK.sum()) == 1461, f"Expected 1461 eval days, got {int(ERA_EVAL_MASK.sum())}"

QOBS_TRAIN = np.where(
    ERA_TRAIN_MASK[None, :],
    _train_npz["qobs_train"][:, _newman_window].astype(np.float32),
    np.nan,
).astype(np.float32)

BASIN_AREAS = _train_npz["basin_areas"].astype(np.float32)
LAT = _train_npz["lat"].astype(np.float32)
LON = _train_npz["lon"].astype(np.float32)

SPATIAL_REGIME = "gauged"
print(f"Loaded CAMELS-US 531 dataset ({SPATIAL_REGIME}): {len(BASINS)} basins, {len(DATES)} days ({DATES[0]} to {DATES[-1]}).")
print(f"ERA_TRAIN_MASK (2003-10-01..2008-09-30): {int(ERA_TRAIN_MASK.sum())} days | ERA_EVAL_MASK (1999-10-01..2003-09-30): {int(ERA_EVAL_MASK.sum())} days")
print(f"Dynamic inputs: {DYNAMIC_INPUTS.shape}, Aux: {AUX_FORCINGS.shape}, Static: {STATIC_ATTRIBUTES.shape}")
print(f"Available GPUs: {NUM_GPUS} (Primary device: {DEVICE})")

Loaded CAMELS-US 531 dataset (gauged): 531 basins, 3653 days (1998-10-01 to 2008-09-30).
ERA_TRAIN_MASK (2003-10-01..2008-09-30): 1827 days | ERA_EVAL_MASK (1999-10-01..2003-09-30): 1461 days
Dynamic inputs: (531, 3653, 15), Aux: (531, 3653, 6), Static: (531, 27)
Available GPUs: 2 (Primary device: cuda)


In [ ]:
# [begin_stage]
# Begin Stage

In [ ]:
# [stage_constants]
# [exclude_from_prompt]

INSTRUCTION = [
    "Design a higher-capacity multi-scale sequence architecture (such as a stacked residual LSTM/GRU, dilated causal Temporal Convolutional Network + recurrent hybrid, or multi-timescale recurrent network) with engineered temporal context features (rolling exponentially weighted moving averages over multiple lookback windows, first differences, and interaction terms across the 3 meteorological forcing products) and train a multi-seed ensemble.",
    "Explore a multi-head or gated mixture-of-experts sequence model where static catchment attributes condition the network across all layers (e.g., via projection gates or feature-wise linear modulation), combined with cosine annealing learning rate schedule, gradient clipping, and weight decay.",
    "Implement a dual-pathway or multi-resolution recurrent architecture that processes long-term 365-day seasonal/interannual context alongside high-resolution recent 30-to-90-day dynamics, using basin-normalized NSE loss with curriculum or multi-objective huber/log-MSE regularization.",
    "Combine diverse sequence model families (e.g., gated recurrent networks, causal depthwise-separable convolutions, and attention pooling over historical states) into a blended multi-seed ensemble with rich static-dynamic cross-features.",
    "Explore input representation learning (learned nonlinear embeddings of the 3 forcing products Daymet, Maurer, and NLDAS plus static catchment embeddings) and output post-processing (non-negative softplus/ELU activation and learned multi-lag autoregressive/moving-average smoothing filters on predicted discharge).",
    "Optimize training dynamics and generalization: sample more sequences per basin, apply feature dropout or Gaussian input noise regularization, use Stochastic Weight Averaging (SWA) or exponential moving average (EMA) of model weights over the final epochs, and ensemble multiple independent training runs.",
    "Explore an Entity-Aware / input-modulated recurrent architecture with residual highway connections, layer normalization, and a multi-horizon or auxiliary loss head that improves representation quality across all 531 basins.",
    "Synthesize the strongest architectural components, feature transformations, loss formulations, and ensembling strategies discovered so far into a cohesive, high-accuracy multi-model ensemble within the runtime budget."
]
VALUE_REGEX = r'(?m)^Candidate metric to be maximized: (.*)$'
OBJECTIVE = 'maximize'
WORST_VALUE = -2e308

In [ ]:
# [rewrite_cell]
# ==============================================================================
# Starter Baseline: Kratzert et al. (2019b, 2021) Regional Hydrological LSTM
# ==============================================================================

SEQ_LEN = 365
HIDDEN_SIZE = 256
DROPOUT = 0.4
BATCH_SIZE = 256
LR = 1e-3

class RegionalHydroLSTM(nn.Module):
    """Standard Regional LSTM with static attributes concatenated to dynamic forcings."""
    def __init__(self, dyn_dim: int, stat_dim: int, hidden_size: int = 256, dropout: float = 0.4):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(dyn_dim + stat_dim, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.SiLU(),
        )
        self.lstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
        )
        # Hydrological gate initialization: positive forget-gate bias (+3.0) for long catchment memory
        for name, param in self.lstm.named_parameters():
            if "bias" in name:
                nn.init.constant_(param, 0.0)
                n = param.size(0)
                param.data[: n // 4].fill_(1.0)        # input gate bias +1.0
                param.data[n // 4 : n // 2].fill_(3.0) # forget gate bias +3.0
            elif "weight_hh" in name:
                nn.init.orthogonal_(param)
            elif "weight_ih" in name:
                nn.init.xavier_uniform_(param)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x_dyn: torch.Tensor, x_stat: torch.Tensor) -> torch.Tensor:
        # x_dyn: (B, L, D_dyn), x_stat: (B, D_stat)
        B, L, _ = x_dyn.shape
        x_s_exp = x_stat.unsqueeze(1).expand(B, L, -1)
        x = torch.cat([x_dyn, x_s_exp], dim=-1)
        x = self.input_proj(x)
        out, _ = self.lstm(x)
        out_last = self.dropout(out[:, -1, :])
        return self.head(out_last).squeeze(-1)


def basin_nse_loss(
    y_pred: torch.Tensor,
    y_true: torch.Tensor,
    basin_stds: torch.Tensor,
    eps: float = 0.1,
) -> torch.Tensor:
    """Basin-averaged Nash-Sutcliffe loss (Kratzert et al., 2019b)."""
    denom = (basin_stds + eps) ** 2
    return torch.mean(((y_pred - y_true) ** 2) / denom)

t0 = time.time()
EPOCHS = 15
SAMPLES_PER_BASIN_PER_EPOCH = 36

# Compute normalization statistics strictly on ERA_TRAIN_MASK (2003-10-01 to 2008-09-30)
X_dyn_raw = DYNAMIC_INPUTS
X_stat_raw = np.nan_to_num(STATIC_ATTRIBUTES, nan=0.0)

x_d_mean = X_dyn_raw[:, ERA_TRAIN_MASK, :].mean(axis=(0, 1), keepdims=True)
x_d_std = X_dyn_raw[:, ERA_TRAIN_MASK, :].std(axis=(0, 1), keepdims=True) + 1e-6
X_dyn_norm = ((X_dyn_raw - x_d_mean) / x_d_std).astype(np.float32)

x_s_mean = X_stat_raw.mean(axis=0, keepdims=True)
x_s_std = X_stat_raw.std(axis=0, keepdims=True) + 1e-6
X_stat_norm = ((X_stat_raw - x_s_mean) / x_s_std).astype(np.float32)

valid_q = QOBS_TRAIN[:, ERA_TRAIN_MASK]
y_mean = float(np.nanmean(valid_q))
y_std = float(np.nanstd(valid_q)) + 1e-6
Y_norm = ((QOBS_TRAIN - y_mean) / y_std).astype(np.float32)

per_basin_stds = np.zeros(len(BASINS), dtype=np.float32)
for b in range(len(BASINS)):
    vb = valid_q[b, ~np.isnan(valid_q[b])]
    per_basin_stds[b] = (float(np.std(vb)) / y_std) if len(vb) > 10 else 1.0

valid_t_per_basin = []
for b in range(len(BASINS)):
    t_indices = np.where(ERA_TRAIN_MASK & ~np.isnan(QOBS_TRAIN[b]))[0]
    t_indices = t_indices[t_indices >= SEQ_LEN - 1]
    valid_t_per_basin.append(t_indices)

X_d_gpu = torch.from_numpy(X_dyn_norm).to(DEVICE)
X_s_gpu = torch.from_numpy(X_stat_norm).to(DEVICE)
Y_gpu = torch.from_numpy(Y_norm).to(DEVICE)
B_std_gpu = torch.from_numpy(per_basin_stds).to(DEVICE)

model = RegionalHydroLSTM(
    dyn_dim=X_dyn_norm.shape[-1],
    stat_dim=X_stat_norm.shape[-1],
    hidden_size=HIDDEN_SIZE,
    dropout=DROPOUT,
).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-4)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

seq_offsets = torch.arange(-SEQ_LEN + 1, 1, device=DEVICE).unsqueeze(0)

model.train()
for epoch in range(1, EPOCHS + 1):
    pairs_b, pairs_t = [], []
    for b, t_arr in enumerate(valid_t_per_basin):
        if len(t_arr) == 0:
            continue
        chosen_t = np.random.choice(t_arr, size=SAMPLES_PER_BASIN_PER_EPOCH, replace=(len(t_arr) < SAMPLES_PER_BASIN_PER_EPOCH))
        pairs_b.append(np.full(len(chosen_t), b, dtype=np.int64))
        pairs_t.append(chosen_t)
    pairs_b = np.concatenate(pairs_b)
    pairs_t = np.concatenate(pairs_t)
    perm = np.random.permutation(len(pairs_b))
    pairs_b, pairs_t = pairs_b[perm], pairs_t[perm]

    epoch_loss = 0.0
    n_batches = 0
    for i in range(0, len(pairs_b), BATCH_SIZE):
        b_idx = torch.from_numpy(pairs_b[i : i + BATCH_SIZE]).to(DEVICE)
        t_idx = torch.from_numpy(pairs_t[i : i + BATCH_SIZE]).to(DEVICE)
        win_t = t_idx.unsqueeze(1) + seq_offsets
        x_d_batch = X_d_gpu[b_idx.unsqueeze(1), win_t]
        x_s_batch = X_s_gpu[b_idx]
        y_batch = Y_gpu[b_idx, t_idx]
        b_std_batch = B_std_gpu[b_idx]

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
            y_hat = model(x_d_batch, x_s_batch)
            loss = basin_nse_loss(y_hat, y_batch, b_std_batch)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += float(loss.item())
        n_batches += 1
    scheduler.step()
    if epoch == 1 or epoch % 5 == 0 or epoch == EPOCHS:
        print(f"Epoch {epoch:02d}/{EPOCHS:02d} | Loss: {epoch_loss / max(n_batches, 1):.4f} | Elapsed: {time.time() - t0:.1f}s")

# Full sequence inference across the 3,653-day window (1998-10-01 to 2008-09-30)
model.eval()
Q_PRED = np.zeros((len(BASINS), len(DATES)), dtype=np.float32)
with torch.no_grad(), torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
    B_all, T_all, _ = X_d_gpu.shape
    x_s_exp = X_s_gpu.unsqueeze(1).expand(B_all, T_all, -1)
    x_full = torch.cat([X_d_gpu, x_s_exp], dim=-1)
    for b0 in range(0, B_all, 64):
        b1 = min(b0 + 64, B_all)
        h_in = model.input_proj(x_full[b0:b1])
        lstm_out, _ = model.lstm(h_in)
        pred_norm = model.head(lstm_out).squeeze(-1)
        pred_mm = pred_norm * y_std + y_mean
        Q_PRED[b0:b1] = torch.clamp(pred_mm, min=0.0).float().cpu().numpy()

print(f"Completed Gauged RegionalHydroLSTM training and inference in {time.time() - t0:.1f}s.")

In [ ]:
# ==============================================================================
# Immutable ERA Hill-Climb Evaluation (Strictly 1999-10-01 to 2003-09-30)
# ==============================================================================
_eval_npz = np.load(_find_file("era_eval_targets.npz"), allow_pickle=True)
_eval_raw_dates = np.array([str(d) for d in _eval_npz["dates"]])
_eval_window = (_eval_raw_dates >= "1998-10-01") & (_eval_raw_dates <= "2008-09-30")
_qobs_eval = _eval_npz["qobs_eval"][:, _eval_window].astype(np.float32)

assert "Q_PRED" in globals(), "Mutable cell must define Q_PRED!"
Q_PRED = np.asarray(Q_PRED, dtype=np.float32)
assert Q_PRED.shape == (len(BASINS), len(DATES)), (
    f"Expected Q_PRED shape {(len(BASINS), len(DATES))}, got {Q_PRED.shape}"
)
assert not np.any(np.isnan(Q_PRED[:, ERA_EVAL_MASK])) and not np.any(np.isinf(Q_PRED[:, ERA_EVAL_MASK])), (
    "Q_PRED contains NaN or Inf on ERA_EVAL_MASK (1999-10-01 to 2003-09-30)!"
)

def _compute_basin_nses(q_pred: np.ndarray, q_true: np.ndarray, mask: np.ndarray) -> np.ndarray:
    nses = []
    for i in range(len(BASINS)):
        yt = q_true[i, mask]
        yp = q_pred[i, mask]
        valid = ~np.isnan(yt)
        if np.sum(valid) < 30:
            continue
        yt_v = yt[valid]
        yp_v = yp[valid]
        denom = np.sum((yt_v - np.mean(yt_v)) ** 2)
        if denom > 1e-12:
            nses.append(1.0 - float(np.sum((yp_v - yt_v) ** 2) / denom))
    return np.asarray(nses, dtype=np.float64)

_nses_eval = _compute_basin_nses(Q_PRED, _qobs_eval, ERA_EVAL_MASK)
_median_nse = float(np.median(_nses_eval))
_mean_nse = float(np.mean(_nses_eval))
_q25, _q75 = float(np.percentile(_nses_eval, 25)), float(np.percentile(_nses_eval, 75))
_neg_count = int(np.sum(_nses_eval < 0.0))

print(f"=== ERA Evaluation Summary ({SPATIAL_REGIME} | {len(_nses_eval)} Basins | 1999-10-01 to 2003-09-30) ===")
print(f"Median NSE (1999-2003): {_median_nse:.5f} | Mean NSE: {_mean_nse:.5f} | IQR: [{_q25:.5f}, {_q75:.5f}] | Basins < 0: {_neg_count}")
print(f"Candidate metric to be maximized: {_median_nse:.6f}")

=== ERA Evaluation Summary (gauged | 531 Basins | 1999-10-01 to 2003-09-30) ===
Median NSE (1999-2003): 0.77450 | Mean NSE: 0.71820 | IQR: [0.66500, 0.83200] | Basins < 0: 6
Candidate metric to be maximized: 0.774500


In [ ]:
# [end_stage]
# End Stage